In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
import time

In [0]:
# 1. Source and Target Tables
source_silver_table = "restaurant_project.silver.rest_sales"
target_dim_products = "restaurant_project.gold.dim_products"
target_dim_date     = "restaurant_project.gold.dim_date"
target_dim_branches = "restaurant_project.gold.dim_branches"
target_fact_orders  = "restaurant_project.gold.fact_orders"

def load_gold_layer():
    try:
        batch_start_time = time.time()
        print('================================================')
        print('Loading Gold Layer (Star Schema)')
        print('================================================')

        # read data from silver layer
        df_silver = spark.read.table(source_silver_table)

        # --- 1. Create Dimension: gold.dim_products ---
        print('------------------------------------------------')
        print('Loading dim_products Table')
        print('------------------------------------------------')
        start_time = time.time()

        dim_products_df = df_silver.groupBy("item_name", "category")\
            .agg(F.round(F.avg("price"), 2).alias("avg_unit_price"))

        # Sequential Key and Column Ordering
        prod_window = Window.orderBy("item_name")
        dim_products_df = dim_products_df.withColumn(
            "product_skey", F.row_number().over(prod_window)
        )

        # Reorder: Key first, then descriptive columns
        dim_products_df = dim_products_df.select(
            "product_skey", "item_name", "category", "avg_unit_price"
        )

        # write dim_products to gold layer
        dim_products_df.write.format("delta").mode("overwrite").saveAsTable(target_dim_products)

        print(f'>> Rows Loaded: {dim_products_df.count()}')
        print(f'>> Load Duration: {round(time.time() - start_time, 2)} seconds')

        # --- 2. Create Dimension: gold.dim_branches ---
        print('------------------------------------------------')
        print('Loading dim_branches Table')
        print('------------------------------------------------')
        start_time = time.time()

        dim_branches_df = df_silver.select("branch").distinct() 

        # Sequential Key and Column Ordering
        branch_window = Window.orderBy("branch")
        dim_branches_df = dim_branches_df.withColumn(
            "branch_skey", F.row_number().over(branch_window)
        )

        # Reorder: Key first, then branch name
        dim_branches_df = dim_branches_df.select("branch_skey", "branch")

        # write dim_branches to gold layer
        dim_branches_df.write.format("delta").mode("overwrite").saveAsTable(target_dim_branches)

        print(f'>> Rows Loaded: {dim_branches_df.count()}')
        print(f'>> Load Duration: {round(time.time() - start_time, 2)} seconds')

        # --- 3. Create Dimension: gold.dim_date ---
        print('------------------------------------------------')
        print('Loading dim_date Table')
        print('------------------------------------------------')
        start_time = time.time()

        dim_date_df = df_silver.select("order_date", "is_weekend").distinct()

        # Extract date parts and Reorder
        dim_date_df = dim_date_df \
            .withColumn("year", F.year("order_date"))\
            .withColumn("month", F.month("order_date"))\
            .withColumn("month_name", F.date_format("order_date", "MMMM"))\
            .withColumn("day", F.dayofmonth("order_date"))\
            .withColumn("day_name", F.date_format("order_date", "EEEE"))

        # Order: Date key first, then hierarchy (Year -> Month -> Day)
        dim_date_df = dim_date_df.select(
            "order_date", "year", "month", "month_name", "day", "day_name", "is_weekend"
        )

        # write dim_date to gold layer
        dim_date_df.write.format("delta").mode("overwrite").saveAsTable(target_dim_date)

        print(f'>> Rows Loaded: {dim_date_df.count()}')
        print(f'>> Load Duration: {round(time.time() - start_time, 2)} seconds')

        # --- 4. Create Fact: gold.fact_orders ---
        print('------------------------------------------------')
        print('Loading fact_orders Table')
        print('------------------------------------------------')
        start_time = time.time()

        # Join to bring in Surrogate Keys
        fact_orders_df = df_silver \
            .join(dim_products_df, "item_name", "left")\
            .join(dim_branches_df, "branch", "left")
    
        # Column Ordering: Keys -> Date/Time -> Metrics (Quantity/Price)
        fact_orders_df = fact_orders_df.select(
            "sales_skey",       # Primary Fact Key
            "product_skey",     # Foreign Key to Product
            "branch_skey",      # Foreign Key to Branch
            "order_date",       # Foreign Key to Date
            "hour",             # Time dimension
            "quantity",         # Metric 1
            "price",            # Metric 2
            "discount",         # Metric 3
            "total_amount"      # Final Metric
        )
    
        # write fact_orders to gold layer
        fact_orders_df.write.format("delta").mode("overwrite").saveAsTable(target_fact_orders)

        print(f'>> Rows Loaded: {fact_orders_df.count()}')
        print(f'>> Load Duration: {round(time.time() - start_time, 2)} seconds')

        print('==========================================')
        print(f'Gold Layer Load Completed in {round(time.time() - batch_start_time, 2)} sec')
        print('==========================================')



    
    except Exception as e:
        print('==========================================')
        print('ERROR OCCURED DURING LOADING SILVER LAYER')
        print(f'Error Message: {str(e)}')
        print('==========================================')

# Execute the pipeline
load_gold_layer()